
# Edited Results 4 — malignant-state program and residual-axis summaries

This notebook rebuilds the three state-level analyses that were present in the former state-identity figure, using the **final revised inputs** used for the revision:

1. developmental-program enrichment across the Burdziak `ductal`, `transitional`, and `malignant_ductal` states;
2. cancer-program enrichment across the same states, using the audited AE2 log1p-CPM program scores;
3. enrichment of the **K1–K6-defined** residual PCs across the same states, using the final tumor-defined residual coordinates from AE2.

The original plots z-scored each score/axis across Burdziak cells and then averaged cells within state. Here that original cell-weighted summary is retained as a diagnostic, but the manuscript-facing result is **sample-equal**: cell-level z-scores are first averaged within each `cancer_state × batch` combination, then those sample means are averaged equally within state. This changes only the weighting of the state summary; it does not redefine the underlying programs or residual axes.

Run this notebook before `Edited_main_figure_panels_v5.ipynb`.

(*note that in the notebook last running AE was labeled WP, work_package*)


In [ ]:

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
import scanpy as sc
from IPython.display import display

DATA_ROOT = Path("/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data")
AUDITED_ROOT = DATA_ROOT / "revision_audited_mlm"
MANIFEST_PATH = AUDITED_ROOT / "manifest.json"
AE2 = AUDITED_ROOT / "analysis" / "ae2_clean_joint_seed_17"
OUTDIR = AUDITED_ROOT / "analysis" / "edited_results_4_malignant_state_programs"
OUTDIR.mkdir(parents=True, exist_ok=True)

with open(MANIFEST_PATH) as f:
    AUDIT = json.load(f)

CANCER_H5AD = Path(AUDIT["cancer_h5ad"])
RESIDUAL_PATH = AE2 / "tumor_defined_residual_scores_K75.csv"
CANCER_PROGRAM_PATH = AE2 / "cancer_program_scores_log1pCPM.csv"
CANCER_PROGRAM_COVERAGE_PATH = AE2 / "cancer_program_marker_coverage.csv"

for p in [CANCER_H5AD, RESIDUAL_PATH, CANCER_PROGRAM_PATH, CANCER_PROGRAM_COVERAGE_PATH]:
    if not Path(p).exists():
        raise FileNotFoundError(p)

A_can = sc.read_h5ad(CANCER_H5AD)
assert A_can.obs_names.is_unique

required_obs = ["batch", "condition", "cancer_state"]
missing = [c for c in required_obs if c not in A_can.obs.columns]
if missing:
    raise KeyError(f"Cancer AnnData is missing required metadata: {missing}")

print("Cancer object:", A_can.shape)
print("Output directory:", OUTDIR)
print(A_can.obs[required_obs].head())


Cancer object: (28131, 15376)
Output directory: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/edited_results_4_malignant_state_programs
                                               batch condition  cancer_state
120703424252788_DACD394_Kate_plus  DACD394_Kate_plus        K1  transitional
120703436605797_DACD394_Kate_plus  DACD394_Kate_plus        K1  transitional
120703436909853_DACD394_Kate_plus  DACD394_Kate_plus        K1  transitional
120703454952236_DACD394_Kate_plus  DACD394_Kate_plus        K1  transitional
120726896892262_DACD394_Kate_plus  DACD394_Kate_plus        K1  transitional


## Final curated developmental programs

In [2]:

DEVELOPMENTAL_PROGRAMS_MOUSE = {
    "foregut_early_epithelial_identity": [
        "Sox2", "Sox9", "Foxa1", "Foxa2", "Hnf1b",
        "Krt8", "Krt18", "Epcam", "Cldn4"
    ],
    "ductal_differentiation": [
        "Krt19", "Krt7", "Sox9", "Cftr", "Muc1",
        "Mmp7", "Slc4a4", "Anxa4", "Cldn10", "Hnf1b"
    ],
    "acinar_differentiation": [
        "Prss1", "Prss2", "Cela2a", "Cela3b",
        "Cpa1", "Cpa2", "Cpb1", "Ctrb1",
        "Reg1", "Reg2", "Pnlip", "Clps", "Amy2a5"
    ],
    "biliary_like_differentiation": [
        "Krt7", "Krt19", "Sox9", "Epcam", "Muc1",
        "Tff1", "Tff2", "Slc4a2", "Anxa4",
        "Cldn4", "Cldn10", "Hnf1b"
    ],
    "proliferation_cell_cycle": [
        "Mki67", "Top2a", "Pcna", "Mcm2", "Mcm3", "Mcm4",
        "Mcm5", "Mcm6", "Mcm7", "Cdk1", "Ccnb1",
        "Ccnb2", "Ube2c", "Birc5"
    ],
    "morphogenesis_epithelial_maturation": [
        "Epcam", "Krt8", "Krt18", "Krt19", "Cdh1",
        "Cldn3", "Cldn4", "Ocln", "Tjp1", "Itga6", "Lamc2"
    ],
    "secretory_digestive_enzyme_program": [
        "Prss1", "Prss2", "Cela2a", "Cela3b",
        "Cpa1", "Cpa2", "Cpb1", "Ctrb1", "Pnlip", "Clps", "Amy2a5"
    ],
    "progenitor_fetal_like_program": [
        "Sox9", "Prom1", "Aldh1a1", "Aldh1a3",
        "Nes", "Tff1", "Tff2", "S100a10", "Lgals1", "Vim"
    ],
    "neuroendocrine_like_peptide_hormone": [
        "Chga", "Chgb", "Scg2", "Scg3", "Scg5",
        "Snap25", "Syt1", "Syt4", "Syt13", "Rab3a", "Rab3b",
        "Neurod1", "Fev", "Isl1", "Pax6", "Nkx2-2",
        "Gip", "Cck", "Gast", "Tph1", "Penk", "Resp18"
    ],
}

STATE_ORDER = ["ductal", "transitional", "malignant_ductal"]
EXPECTED_SAMPLE_COUNTS = {"ductal": 4, "transitional": 11, "malignant_ductal": 6}

# Pre-specified compact subset for the main Figure 4 cancer-program panel.
# The complete 23-program sample-equal matrix is also written for Supplement/inspection.
MAIN_CANCER_PROGRAMS = [
    "epithelial_identity",
    "emt_invasion",
    "kras_mapk_signaling",
    "cell_cycle",
    "glycolysis_metabolic_rewiring",
    "hypoxia",
    "oxidative_stress",
    "unfolded_protein_response",
    "senescence_sasp",
    "ap1_immediate_early_response",
    "stemness_progenitor_like",
    "therapy_resistance_stress_tolerance",
]


## Helpers: audited row alignment, log1p-CPM scoring, and sample-equal summaries

In [3]:

def align_to_obs(df, obs_names, label):
    """Require explicit cell-ID agreement; never silently repair by row order."""
    out = df.copy()
    out.index = out.index.astype(str)
    target = pd.Index(obs_names.astype(str))
    if out.index.equals(target):
        return out
    if out.index.is_unique and len(out) == len(target) and set(out.index) == set(target):
        out = out.reindex(target)
        assert out.index.equals(target)
        return out
    raise ValueError(
        f"{label}: cell IDs do not match the audited cancer AnnData. "
        f"rows={len(out)}, target={len(target)}, overlap={len(out.index.intersection(target))}"
    )


def score_programs_log1p_cpm(adata, programs, target_sum=1e4, min_genes=3, gene_symbol_col="symbol"):
    """Mean log1p(CPM-like) expression across present genes; matches the audited scoring convention."""
    X = adata.X.tocsr() if sparse.issparse(adata.X) else np.asarray(adata.X)
    symbols = (adata.var[gene_symbol_col].astype(str).to_numpy()
               if gene_symbol_col in adata.var.columns
               else adata.var_names.astype(str).to_numpy())

    gene_to_idx = {}
    for i, g in enumerate(symbols):
        gene_to_idx.setdefault(g, i)

    totals = (np.asarray(X.sum(axis=1)).ravel() if sparse.issparse(X)
              else np.asarray(X).sum(axis=1))
    if np.any(totals < 0):
        raise ValueError("Negative library totals: expected the audited non-negative aligned count matrix.")
    scale = target_sum / np.maximum(totals, 1.0)

    scores = {}
    coverage = []
    for pname, genes in programs.items():
        present = [g for g in genes if g in gene_to_idx]
        coverage.append({
            "program": pname,
            "n_total": len(genes),
            "n_present": len(present),
            "fraction_present": len(present) / max(len(genes), 1),
        })
        if len(present) < min_genes:
            warnings.warn(f"Skipping {pname}: only {len(present)} genes present")
            continue
        idx = [gene_to_idx[g] for g in present]
        Xi = X[:, idx].toarray() if sparse.issparse(X) else X[:, idx]
        Xi = np.log1p(Xi * scale[:, None])
        scores[pname] = Xi.mean(axis=1)

    return pd.DataFrame(scores, index=adata.obs_names.astype(str)), pd.DataFrame(coverage)


def zscore_columns(df):
    out = df.astype(float).copy()
    for col in out.columns:
        mu = out[col].mean()
        sd = out[col].std(ddof=1)
        out[col] = 0.0 if (not np.isfinite(sd) or sd == 0) else (out[col] - mu) / sd
    return out


def summarize_by_state_sample_equal(values, metadata, prefix):
    """
    Original logic: z-score each feature across all Burdziak cells.
    Revised weighting: cell z-scores -> sample mean within state -> equal-sample state mean.
    """
    values = align_to_obs(values, metadata.index.astype(str), prefix)
    z = zscore_columns(values)
    z["cancer_state"] = metadata["cancer_state"].astype(str).to_numpy()
    z["batch"] = metadata["batch"].astype(str).to_numpy()

    z = z[z["cancer_state"].isin(STATE_ORDER)].copy()
    feature_cols = list(values.columns)

    sample = (
        z.groupby(["cancer_state", "batch"], observed=True)[feature_cols]
         .mean()
         .reset_index()
    )
    state_equal = (
        sample.groupby("cancer_state", observed=True)[feature_cols]
              .mean()
              .reindex(STATE_ORDER)
    )
    state_cellweighted = (
        z.groupby("cancer_state", observed=True)[feature_cols]
         .mean()
         .reindex(STATE_ORDER)
    )
    sample_counts = sample.groupby("cancer_state", observed=True)["batch"].nunique().reindex(STATE_ORDER)

    return z, sample, state_equal, state_cellweighted, sample_counts


## Integrity checks and final state/sample bookkeeping

In [4]:

meta = A_can.obs[["batch", "condition", "cancer_state"]].copy()
meta.index = meta.index.astype(str)

state_counts = meta["cancer_state"].astype(str).value_counts().reindex(STATE_ORDER)
print("Cell counts by state:")
display(state_counts.to_frame("n_cells"))

sample_counts = (
    meta.assign(cancer_state=meta["cancer_state"].astype(str), batch=meta["batch"].astype(str))
        .query("cancer_state in @STATE_ORDER")
        .groupby("cancer_state", observed=True)["batch"].nunique()
        .reindex(STATE_ORDER)
)
print("Distinct Burdziak sample labels represented in each state:")
display(sample_counts.to_frame("n_samples"))

for state, expected in EXPECTED_SAMPLE_COUNTS.items():
    observed = int(sample_counts.loc[state])
    assert observed == expected, f"{state}: expected {expected} sample labels, found {observed}"

sample_counts.rename("n_samples").to_csv(OUTDIR / "edited_results_4_sample_counts_by_state.csv")


Cell counts by state:


,n_cells
cancer_state,
ductal,5626
transitional,13650
malignant_ductal,8855


Distinct Burdziak sample labels represented in each state:


,n_samples
cancer_state,
ductal,4
transitional,11
malignant_ductal,6


## A. Recompute developmental-program scores in the audited Burdziak expression matrix

In [5]:

dev_scores, dev_coverage = score_programs_log1p_cpm(
    A_can,
    DEVELOPMENTAL_PROGRAMS_MOUSE,
    target_sum=1e4,
    min_genes=3,
    gene_symbol_col="symbol",
)

dev_scores = align_to_obs(dev_scores, A_can.obs_names.astype(str), "developmental program scores")
dev_coverage.to_csv(OUTDIR / "edited_results_4_developmental_program_marker_coverage.csv", index=False)
dev_scores.to_csv(OUTDIR / "edited_results_4_developmental_program_scores_log1pCPM.csv.gz", compression="gzip")

print("Developmental program marker coverage:")
display(dev_coverage)
assert set(DEVELOPMENTAL_PROGRAMS_MOUSE).issubset(dev_scores.columns)


Developmental program marker coverage:


,program,n_total,n_present,fraction_present
0,foregut_early_epithelial_identity,9,9,1.0
1,ductal_differentiation,10,10,1.0
2,acinar_differentiation,13,13,1.0
3,biliary_like_differentiation,12,12,1.0
4,proliferation_cell_cycle,14,14,1.0
5,morphogenesis_epithelial_maturation,11,11,1.0
6,secretory_digestive_enzyme_program,11,11,1.0
7,progenitor_fetal_like_program,10,10,1.0
8,neuroendocrine_like_peptide_hormone,22,22,1.0


In [6]:

_, dev_sample, dev_state_equal, dev_state_cellweighted, dev_n = summarize_by_state_sample_equal(
    dev_scores, meta, "developmental programs"
)

dev_sample.to_csv(OUTDIR / "edited_results_4_sample_developmental_program_zmeans.csv", index=False)
dev_state_equal.to_csv(OUTDIR / "edited_results_4_state_developmental_program_zmeans_sample_equal.csv")
dev_state_cellweighted.to_csv(OUTDIR / "edited_results_4_state_developmental_program_zmeans_cell_weighted.csv")

print("Sample-equal developmental-program state matrix:")
display(dev_state_equal.round(3))
print("Max absolute change vs cell-weighted old-style state summary:",
      float((dev_state_equal - dev_state_cellweighted).abs().to_numpy().max()))


Sample-equal developmental-program state matrix:


,foregut_early_epithelial_identity,ductal_differentiation,acinar_differentiation,biliary_like_differentiation,proliferation_cell_cycle,morphogenesis_epithelial_maturation,secretory_digestive_enzyme_program,progenitor_fetal_like_program,neuroendocrine_like_peptide_hormone
cancer_state,,,,,,,,,
ductal,-1.205,-1.312,1.542,-1.206,-0.331,-1.157,1.560,-1.058,-0.255
transitional,0.420,0.303,-0.125,0.395,-0.140,0.403,-0.167,0.003,0.273
malignant_ductal,0.293,0.471,-0.879,0.320,0.506,0.283,-0.829,0.646,-0.250


Max absolute change vs cell-weighted old-style state summary: 0.15260536597039054


## B. Recompile audited AE2 cancer-program scores as sample-equal state enrichments

In [ ]:

cancer_scores = pd.read_csv(CANCER_PROGRAM_PATH, index_col=0)
cancer_scores = align_to_obs(cancer_scores, A_can.obs_names.astype(str), AE2 cancer program scores")
cancer_coverage = pd.read_csv(CANCER_PROGRAM_COVERAGE_PATH)

missing_main = [p for p in MAIN_CANCER_PROGRAMS if p not in cancer_scores.columns]
if missing_main:
    raise KeyError(f"Main Figure 4 cancer programs absent from AE2 scores: {missing_main}")

_, cancer_sample, cancer_state_equal, cancer_state_cellweighted, cancer_n = summarize_by_state_sample_equal(
    cancer_scores, meta, "cancer programs"
)

cancer_sample.to_csv(OUTDIR / "edited_results_4_sample_cancer_program_zmeans.csv", index=False)
cancer_state_equal.to_csv(OUTDIR / "edited_results_4_state_cancer_program_zmeans_sample_equal_full.csv")
cancer_state_cellweighted.to_csv(OUTDIR / "edited_results_4_state_cancer_program_zmeans_cell_weighted_full.csv")
cancer_state_equal[MAIN_CANCER_PROGRAMS].to_csv(
    OUTDIR / "edited_results_4_state_cancer_program_zmeans_sample_equal_main.csv"
)

print("Main-panel cancer-program subset (sample-equal):")
display(cancer_state_equal[MAIN_CANCER_PROGRAMS].round(3))
print("Max absolute change vs cell-weighted old-style state summary:",
      float((cancer_state_equal - cancer_state_cellweighted).abs().to_numpy().max()))


Main-panel cancer-program subset (sample-equal):


,epithelial_identity,emt_invasion,kras_mapk_signaling,cell_cycle,glycolysis_metabolic_rewiring,hypoxia,oxidative_stress,unfolded_protein_response,senescence_sasp,ap1_immediate_early_response,stemness_progenitor_like,therapy_resistance_stress_tolerance
cancer_state,,,,,,,,,,,,
ductal,-1.191,-0.755,-1.177,-0.308,-1.177,-1.061,-1.318,-1.068,-0.965,-0.993,-1.331,-1.244
transitional,0.377,0.038,0.403,-0.147,-0.119,-0.032,0.190,0.403,0.145,0.453,0.154,0.186
malignant_ductal,0.355,0.272,0.279,0.488,0.940,0.749,0.643,0.269,0.377,0.076,0.658,0.577


Max absolute change vs cell-weighted old-style state summary: 0.33405828507373003


## C. Recompile the final K1–K6-defined residual PCs as sample-equal state enrichments

In [ ]:

res = pd.read_csv(RESIDUAL_PATH, index_col=0)
res = align_to_obs(res, A_can.obs_names.astype(str), "AE2 tumor-defined residual coordinates")

# Confirm that the residual table carries exactly the audited metadata after alignment.
for col in ["batch", "condition", "cancer_state"]:
    if col not in res.columns:
        raise KeyError(f"Residual score table lacks {col}")
    lhs = res[col].astype(str).to_numpy()
    rhs = meta[col].astype(str).to_numpy()
    assert np.array_equal(lhs, rhs), f"Residual metadata mismatch in {col}"

pc_cols = [f"TumorResPC{i}" for i in range(1, 11)]
missing_pcs = [c for c in pc_cols if c not in res.columns]
if missing_pcs:
    raise KeyError(f"Missing residual PCs: {missing_pcs}")

res_values = res[pc_cols].copy()
_, res_sample, res_state_equal, res_state_cellweighted, res_n = summarize_by_state_sample_equal(
    res_values, meta, "K-series-defined residual PCs"
)

res_sample.to_csv(OUTDIR / "edited_results_4_sample_residual_pc_zmeans.csv", index=False)
res_state_equal.to_csv(OUTDIR / "edited_results_4_state_residual_pc_zmeans_sample_equal.csv")
res_state_cellweighted.to_csv(OUTDIR / "edited_results_4_state_residual_pc_zmeans_cell_weighted.csv")

print("Sample-equal residual-PC state matrix:")
display(res_state_equal.round(3))
print("Max absolute change vs cell-weighted old-style state summary:",
      float((res_state_equal - res_state_cellweighted).abs().to_numpy().max()))


Sample-equal residual-PC state matrix:


,TumorResPC1,TumorResPC2,TumorResPC3,TumorResPC4,TumorResPC5,TumorResPC6,TumorResPC7,TumorResPC8,TumorResPC9,TumorResPC10
cancer_state,,,,,,,,,,
ductal,0.076,0.961,-0.156,-0.339,0.362,1.012,0.027,-0.303,-0.438,0.669
transitional,0.576,-0.394,-0.160,-0.273,-0.052,-0.200,-0.058,-0.068,0.033,-0.047
malignant_ductal,-0.856,-0.088,0.513,0.618,0.181,-0.145,0.153,0.589,0.114,-0.633


Max absolute change vs cell-weighted old-style state summary: 0.29612366078744595


## Final audit and manifest

In [ ]:

expected_outputs = [
    "edited_results_4_sample_counts_by_state.csv",
    "edited_results_4_developmental_program_marker_coverage.csv",
    "edited_results_4_developmental_program_scores_log1pCPM.csv.gz",
    "edited_results_4_sample_developmental_program_zmeans.csv",
    "edited_results_4_state_developmental_program_zmeans_sample_equal.csv",
    "edited_results_4_state_developmental_program_zmeans_cell_weighted.csv",
    "edited_results_4_sample_cancer_program_zmeans.csv",
    "edited_results_4_state_cancer_program_zmeans_sample_equal_full.csv",
    "edited_results_4_state_cancer_program_zmeans_cell_weighted_full.csv",
    "edited_results_4_state_cancer_program_zmeans_sample_equal_main.csv",
    "edited_results_4_sample_residual_pc_zmeans.csv",
    "edited_results_4_state_residual_pc_zmeans_sample_equal.csv",
    "edited_results_4_state_residual_pc_zmeans_cell_weighted.csv",
]

missing = [name for name in expected_outputs if not (OUTDIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing expected outputs: {missing}")

assert list(dev_state_equal.index) == STATE_ORDER
assert list(cancer_state_equal.index) == STATE_ORDER
assert list(res_state_equal.index) == STATE_ORDER
assert dev_state_equal.shape[1] == len(DEVELOPMENTAL_PROGRAMS_MOUSE)
assert res_state_equal.shape[1] == 10
assert np.isfinite(dev_state_equal.to_numpy()).all()
assert np.isfinite(cancer_state_equal[MAIN_CANCER_PROGRAMS].to_numpy()).all()
assert np.isfinite(res_state_equal.to_numpy()).all()

manifest = {
    "status": "PASS",
    "purpose": "Recompute former malignant-state program/residual panels with audited inputs and sample-equal state weighting.",
    "cancer_h5ad": str(CANCER_H5AD),
    "ae2_cancer_program_scores": str(CANCER_PROGRAM_PATH),
    "ae2_tumor_defined_residual_scores": str(RESIDUAL_PATH),
    "state_order": STATE_ORDER,
    "sample_counts": {k: int(v) for k, v in sample_counts.items()},
    "developmental_programs": list(DEVELOPMENTAL_PROGRAMS_MOUSE),
    "main_cancer_programs": MAIN_CANCER_PROGRAMS,
    "residual_pcs": pc_cols,
    "zscore_scope": "all audited Burdziak cells",
    "state_weighting": "cell z-scores -> mean within cancer_state x batch -> equal mean across batch labels within state",
    "outputs": [str(OUTDIR / name) for name in expected_outputs],
}
with open(OUTDIR / "edited_results_4_audit_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("EDITED RESULTS 4 AUDIT STATUS: PASS")
print("Wrote:", OUTDIR / "edited_results_4_audit_manifest.json")


EDITED RESULTS 4 AUDIT STATUS: PASS
Wrote: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/edited_results_4_malignant_state_programs/edited_results_4_audit_manifest.json
